# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrathamTumminakatti/ML1/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

# 1. Method Choice and Why

## Method: Logistic Regression

I chose Logistic Regression as the first modeling method for the Content Refresh lane because the goal is to create an explainable model that can support a refresh-priority decision.

The model can estimate the probability that a content item should be prioritized for refresh based on measurable content and performance signals.

Logistic Regression is appropriate as a baseline model because it is:

- Simple and interpretable
- Fast to train
- Suitable for binary classification
- Useful for understanding which signals are associated with the predicted action

I will compare the model against my Week-4 rule-based baseline using the same data, split, and evaluation metric.

The model is intended as decision-support, not as proof that a page will improve after a refresh.




## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

# 2. Split Design

I will use a grouped train/test split based on `client_id`.

The client ID will be used only for creating the groups, not as a model feature. This keeps content from the same client together in either the training or test set.

This split is more honest for the Content Refresh question because content from the same client can share characteristics such as publishing practices, traffic patterns, and content mix. Keeping clients separated reduces the risk that the model benefits from seeing very similar client-specific patterns during training.

I will use an 80/20 train/test split with a fixed random state so the result is reproducible.

The test set will be kept separate until evaluation.

In [22]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

# Use the same dataset as the Week-4 baseline
data = pd.read_csv("content_refresh_anonymized.csv")
model_data = data.copy()

# Grouped split: keep each client entirely in train or test
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_data,
        groups=model_data["client_id"]
    )
)

train_data = model_data.iloc[train_idx].copy()
test_data = model_data.iloc[test_idx].copy()

print("Train rows:", len(train_data))
print("Test rows:", len(test_data))
print("Train clients:", train_data["client_id"].nunique())
print("Test clients:", test_data["client_id"].nunique())

# Verify that no client appears in both sets
overlap = set(train_data["client_id"]) & set(test_data["client_id"])

print("Client overlap:", len(overlap))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

# 3. Train + Compare vs My Baseline

I will train a Logistic Regression model to reproduce the Week-4 baseline refresh-priority decision.

The model uses the same underlying content and performance data while excluding identifiers and leakage-prone trend variables.

I will evaluate the model and the Week-4 baseline on the same held-out test set.

The comparison is intended to measure how closely the model can reproduce the baseline decision and whether the model provides a more structured and interpretable way to rank refresh candidates.

This is a decision-support comparison, not a claim that the model predicts the real-world success of a content refresh.

In [23]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ---------------------------------------------------------
# 1. Recreate the Week-4 baseline
# ---------------------------------------------------------

baseline = data.copy()

baseline["age_score"] = (
    baseline["content_age_days"]
    / baseline["content_age_days"].max()
)

baseline["stale_score"] = (
    baseline["days_since_last_update"]
    / baseline["days_since_last_update"].max()
)

baseline["impression_score"] = (
    baseline["impressions_90d"]
    / baseline["impressions_90d"].max()
)

baseline["ctr_opportunity"] = 1 - (
    baseline["ctr"] / baseline["ctr"].max()
)

baseline["baseline_score"] = (
    0.35 * baseline["age_score"]
    + 0.30 * baseline["stale_score"]
    + 0.25 * baseline["impression_score"]
    + 0.10 * baseline["ctr_opportunity"]
)

# Attach baseline score to train/test using their original indices
train_data["baseline_score"] = baseline.loc[
    train_data.index, "baseline_score"
]

test_data["baseline_score"] = baseline.loc[
    test_data.index, "baseline_score"
]

print("Baseline score recreated successfully.")


# ---------------------------------------------------------
# 2. Create refresh-priority target
# ---------------------------------------------------------

threshold = train_data["baseline_score"].quantile(0.80)

train_data["refresh_target"] = (
    train_data["baseline_score"] >= threshold
).astype(int)

test_data["refresh_target"] = (
    test_data["baseline_score"] >= threshold
).astype(int)

print("Refresh-priority threshold:", round(threshold, 4))


# ---------------------------------------------------------
# 3. Define model features
# ---------------------------------------------------------

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

# Keep only columns that actually exist
numeric_features = [
    col for col in numeric_features
    if col in train_data.columns
]

categorical_features = [
    col for col in categorical_features
    if col in train_data.columns
]

features = numeric_features + categorical_features


# ---------------------------------------------------------
# 4. Prepare train/test data
# ---------------------------------------------------------

X_train = train_data[features]
y_train = train_data["refresh_target"]

X_test = test_data[features]
y_test = test_data["refresh_target"]


# ---------------------------------------------------------
# 5. Preprocessing
# ---------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


# ---------------------------------------------------------
# 6. Logistic Regression
# ---------------------------------------------------------

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)


# ---------------------------------------------------------
# 7. Model evaluation
# ---------------------------------------------------------

model_metrics = {
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision": precision_score(
        y_test,
        predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        predictions,
        zero_division=0
    )
}

print("\nLogistic Regression results:")

for metric, value in model_metrics.items():
    print(f"{metric}: {value:.3f}")

Baseline score recreated successfully.
Refresh-priority threshold: 0.3887

Logistic Regression results:
Accuracy: 0.983
Precision: 0.948
Recall: 1.000
F1: 0.973


In [24]:
import numpy as np

# Recreate the Week-4 baseline score
baseline = data.copy()

# Normalize signals
baseline["age_score"] = (
    baseline["content_age_days"]
    / baseline["content_age_days"].max()
)

baseline["stale_score"] = (
    baseline["days_since_last_update"]
    / baseline["days_since_last_update"].max()
)

baseline["impression_score"] = (
    baseline["impressions_90d"]
    / baseline["impressions_90d"].max()
)

# Lower CTR = greater refresh opportunity
baseline["ctr_opportunity"] = 1 - (
    baseline["ctr"] / baseline["ctr"].max()
)

# Same Week-4 baseline formula
baseline["baseline_score"] = (
    0.35 * baseline["age_score"] +
    0.30 * baseline["stale_score"] +
    0.25 * baseline["impression_score"] +
    0.10 * baseline["ctr_opportunity"]
)

# Add the score back to the train/test data
train_data["baseline_score"] = baseline.loc[
    train_data.index, "baseline_score"
]

test_data["baseline_score"] = baseline.loc[
    test_data.index, "baseline_score"
]

print("Baseline score recreated successfully.")
print(train_data["baseline_score"].describe())

Baseline score recreated successfully.
count    23837.000000
mean         0.294523
std          0.087728
min          0.075938
25%          0.201673
50%          0.296674
75%          0.377968
max          0.643304
Name: baseline_score, dtype: float64


In [25]:
# Compare the model with the Week-4 baseline on the same test set

# Week-4 baseline decision:
# Items at or above the training 80th-percentile threshold are prioritized.
baseline_predictions = (
    test_data["baseline_score"] >= threshold
).astype(int)

baseline_metrics = {
    "Accuracy": accuracy_score(y_test, baseline_predictions),
    "Precision": precision_score(
        y_test,
        baseline_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        baseline_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        baseline_predictions,
        zero_division=0
    )
}

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Week-4 Baseline": [
        baseline_metrics["Accuracy"],
        baseline_metrics["Precision"],
        baseline_metrics["Recall"],
        baseline_metrics["F1"]
    ],
    "Logistic Regression": [
        model_metrics["Accuracy"],
        model_metrics["Precision"],
        model_metrics["Recall"],
        model_metrics["F1"]
    ]
})

comparison

,Metric,Week-4 Baseline,Logistic Regression
0,Accuracy,1.0,0.982963
1,Precision,1.0,0.948097
2,Recall,1.0,1.000000
3,F1,1.0,0.973357


### Comparison

The Week-4 baseline achieved 1.000 on all four metrics because the evaluation target was constructed directly from the baseline score. Therefore, these perfect baseline metrics should not be interpreted as evidence of real-world predictive performance.

Logistic Regression achieved 0.983 accuracy, 0.948 precision, 1.000 recall, and 0.973 F1 on the held-out client groups.

The model closely reproduced the baseline decisions, especially in recall. However, this comparison does not show that the model is better at predicting whether a content refresh will actually succeed. It only shows how closely the learned model approximates the existing decision rule.

For this reason, the result should be treated as decision-support evidence rather than proof of future refresh performance.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [26]:
# Error analysis

# Create a boolean mask for incorrect predictions
error_mask = (y_test.to_numpy() != predictions)

# Select only the incorrectly predicted rows
error_analysis = test_data.iloc[error_mask].copy()

# Add the matching actual and predicted values
error_analysis["actual"] = y_test.to_numpy()[error_mask]
error_analysis["predicted"] = predictions[error_mask]

print("Total test cases:", len(test_data))
print("Incorrect predictions:", len(error_analysis))
print(
    "Error rate:",
    f"{len(error_analysis) / len(test_data):.3f}"
)

print("\nPrediction counts:")
print(pd.Series(predictions).value_counts().sort_index())

Total test cases: 6163
Incorrect predictions: 105
Error rate: 0.017

Prediction counts:
0    4140
1    2023
Name: count, dtype: int64


In [27]:
# Inspect Logistic Regression feature importance

# Get the fitted preprocessing and classifier
preprocessor_fitted = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

# Get transformed feature names
feature_names = preprocessor_fitted.get_feature_names_out()

# Get Logistic Regression coefficients
coefficients = classifier.coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
})

# Strongest positive signals
top_positive = feature_importance.sort_values(
    "coefficient",
    ascending=False
).head(10)

# Strongest negative signals
top_negative = feature_importance.sort_values(
    "coefficient",
    ascending=True
).head(10)

print("Top positive signals:")
display(top_positive[["feature", "coefficient"]])

print("\nTop negative signals:")
display(top_negative[["feature", "coefficient"]])

Top positive signals:


,feature,coefficient
21,num__content_age_days,21.655413
22,num__days_since_last_update,9.716457
5,num__impressions_90d,1.640700
56,cat__impression_tier_excellent,1.233285
45,cat__freshness_tier_181+,1.170376
4,num__char_count,1.116551
28,cat__content_type_keyword article,0.761768
64,cat__position_tier_top_3,0.619084
48,cat__word_count_tier_1000-2000,0.599671
39,cat__model_used_unknown,0.516813



Top negative signals:


,feature,coefficient
36,cat__model_used_gemini-3-flash-preview,-1.130422
27,cat__content_type_feedly article,-1.129971
3,num__word_count,-0.953285
59,cat__impression_tier_moderate,-0.785754
47,cat__freshness_tier_91-180,-0.784499
58,cat__impression_tier_low,-0.708951
53,cat__char_count_tier_25000+,-0.704405
50,cat__word_count_tier_3500+,-0.683216
40,cat__age_tier_181-365,-0.666823
46,cat__freshness_tier_31-90,-0.629709


# 4. Errors and Interpretation

## Error Analysis

The Logistic Regression model made 105 incorrect predictions out of 6,163 test cases, giving an observed error rate of 1.7%.

The model predicted 2,023 items as refresh-priority and 4,140 as not refresh-priority.

The errors show that the model does not reproduce the Week-4 rule perfectly. Some items receive different decisions because the Logistic Regression model combines several signals rather than applying the exact weighted formula used by the baseline.

## Feature Interpretation

The strongest positive signals observed in the fitted model were:

- `content_age_days`
- `days_since_last_update`
- `impressions_90d`
- `impression_tier_excellent`
- `freshness_tier_181+`

This indicates that the model leaned strongly on content age and time since the last update, which is consistent with the purpose of the Content Refresh lane.

Other positive signals included character count, keyword article content type, top-3 position, and the 1000-2000 word-count tier.

The strongest negative coefficients included:

- `model_used_gemini-3-flash-preview`
- `content_type_feedly article`
- `word_count`
- `impression_tier_moderate`
- `freshness_tier_91-180`

These coefficients describe associations within this fitted model and should not be interpreted as causal effects.

## Interpretation

The model closely approximated the Week-4 baseline, with an observed F1 score of 0.973 and recall of 1.000 on the held-out client groups.

The strongest positive signals were content age and days since the last update, suggesting that the learned model captured the main staleness pattern built into the baseline.

However, because the target itself was derived from the Week-4 baseline score, this experiment measures agreement with the existing rule rather than real-world refresh success.

Therefore, the model should be treated as decision-support evidence. A future evaluation with an independently observed refresh outcome would be needed to determine whether the model improves actual refresh decisions.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.